# Notebook 07: Adaptive Density Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase1/07_adaptive_density_control.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand why adaptive density control is crucial for 3DGS
2. Learn when to split, clone, and prune Gaussians
3. Implement densification strategies (splitting and cloning)
4. Implement pruning based on opacity and size
5. Visualize the effects of adaptive control on reconstruction quality

**Estimated Time**: 75 minutes

**Prerequisites**: Notebook 06 (Spherical Harmonics)

---

## Setup

In [ ]:
import os
import sys

# Colab setup
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('3DGS-from-scratch'):
        !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    os.chdir('3DGS-from-scratch')
    !pip install -q plotly ipywidgets

# Path setup
for path in ['../../src', '../src', './src']:
    full_path = os.path.abspath(path)
    if os.path.exists(os.path.join(full_path, 'gaussian')):
        sys.path.insert(0, full_path)
        break

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.patches as mpatches

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("Setup complete!")

## 1. Why Adaptive Density Control?

### The Challenge

When training 3DGS:
- **Initial Gaussians** from point cloud may be poorly distributed
- Some regions need **more detail** (e.g., edges, textures)
- Some Gaussians are **redundant** or **invisible**
- Fixed number of Gaussians cannot adapt to scene complexity

### The Solution: Adaptive Control

During training, we dynamically adjust the Gaussian population:

1. **Densification** - Add Gaussians where needed:
   - **Splitting**: Large Gaussians with high gradients → split into smaller ones
   - **Cloning**: Small Gaussians with high gradients → duplicate and move

2. **Pruning** - Remove unnecessary Gaussians:
   - Very low opacity (α < threshold)
   - Very large in world space
   - Very large on screen (2D extent > threshold)

In [ ]:
# Visualize the adaptive control concept
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Initial state
ax = axes[0]
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.add_patch(Ellipse((0, 0), 2.0, 1.5, angle=30, fill=True, alpha=0.5, color='blue'))
ax.add_patch(Ellipse((1, 0.5), 0.3, 0.2, angle=-20, fill=True, alpha=0.3, color='red'))
ax.add_patch(Ellipse((-1, -0.5), 0.4, 0.3, angle=45, fill=True, alpha=0.1, color='green'))
ax.set_title('Initial State\n(Various sizes & opacities)')
ax.set_aspect('equal')

# After densification
ax = axes[1]
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
# Split large Gaussian
for i, (dx, dy) in enumerate([(-0.4, -0.3), (0.4, 0.3)]):
    ax.add_patch(Ellipse((dx, dy), 0.7, 0.5, angle=30, fill=True, alpha=0.5, color='blue'))
# Clone small Gaussian with high gradient
ax.add_patch(Ellipse((1, 0.5), 0.3, 0.2, angle=-20, fill=True, alpha=0.3, color='red'))
ax.add_patch(Ellipse((1.3, 0.7), 0.3, 0.2, angle=-20, fill=True, alpha=0.3, color='red'))
# Low opacity stays (for now)
ax.add_patch(Ellipse((-1, -0.5), 0.4, 0.3, angle=45, fill=True, alpha=0.1, color='green'))
ax.set_title('After Densification\n(Split large, Clone small)')
ax.set_aspect('equal')

# After pruning
ax = axes[2]
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
for i, (dx, dy) in enumerate([(-0.4, -0.3), (0.4, 0.3)]):
    ax.add_patch(Ellipse((dx, dy), 0.7, 0.5, angle=30, fill=True, alpha=0.5, color='blue'))
ax.add_patch(Ellipse((1, 0.5), 0.3, 0.2, angle=-20, fill=True, alpha=0.3, color='red'))
ax.add_patch(Ellipse((1.3, 0.7), 0.3, 0.2, angle=-20, fill=True, alpha=0.3, color='red'))
# Green one removed (low opacity)
ax.annotate('', xy=(-1, -0.5), xytext=(-1.5, -1.2),
           arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(-1.5, -1.4, 'Pruned\n(low α)', ha='center', fontsize=9, color='red')
ax.set_title('After Pruning\n(Remove low opacity)')
ax.set_aspect('equal')

plt.suptitle('Adaptive Density Control Workflow', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Gradient-Based Densification Criterion

### Key Insight

Gaussians that need densification have **large positional gradients**:

$$\bar{\nabla_\mu} = \frac{1}{T} \sum_{t=1}^{T} \left\| \frac{\partial L}{\partial \mu_{2D}} \right\|$$

where:
- $L$ is the reconstruction loss
- $\mu_{2D}$ is the projected 2D position
- $T$ is number of training iterations

### Why Gradients?

- Large gradient = Gaussian wants to move significantly
- This indicates **under-reconstruction** in that region
- Solution: add more Gaussians to better cover the area

In [ ]:
class GradientAccumulator:
    """
    Accumulates positional gradients over training iterations.
    
    Used to identify Gaussians that need densification.
    """
    
    def __init__(self, n_gaussians: int):
        self.n_gaussians = n_gaussians
        self.gradient_sum = torch.zeros(n_gaussians)
        self.count = torch.zeros(n_gaussians)
    
    def accumulate(self, means_2d: torch.Tensor, visible_mask: torch.Tensor = None):
        """
        Accumulate gradients after backward pass.
        
        Args:
            means_2d: Projected 2D means with gradients [N, 2]
            visible_mask: Which Gaussians were rendered
        """
        if means_2d.grad is None:
            return
        
        grad_norm = means_2d.grad.norm(dim=-1)  # [N]
        
        if visible_mask is not None:
            grad_norm = grad_norm * visible_mask.float()
            self.count += visible_mask.float()
        else:
            self.count += 1
        
        self.gradient_sum += grad_norm.detach()
    
    def get_average_gradients(self) -> torch.Tensor:
        """Get average gradient norm per Gaussian."""
        return self.gradient_sum / (self.count + 1e-8)
    
    def reset(self):
        """Reset accumulators."""
        self.gradient_sum.zero_()
        self.count.zero_()


# Demo: simulate gradient accumulation
N = 100
n_iters = 50

accumulator = GradientAccumulator(N)

# Simulate: some Gaussians consistently have high gradients
torch.manual_seed(42)
for i in range(n_iters):
    means_2d = torch.randn(N, 2, requires_grad=True)
    
    # Simulate loss and backward
    # Some Gaussians get higher gradients (under-reconstructed regions)
    base_loss = (means_2d ** 2).sum()
    
    # Simulate varying gradient magnitudes
    weights = torch.ones(N)
    weights[:20] = 5.0  # First 20 Gaussians have high gradients
    weights[20:40] = 0.1  # Next 20 have low gradients
    
    loss = (weights.unsqueeze(1) * means_2d ** 2).sum()
    loss.backward()
    
    accumulator.accumulate(means_2d)

avg_grads = accumulator.get_average_gradients()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(N), avg_grads.numpy())
axes[0].axhline(y=avg_grads.mean().item(), color='r', linestyle='--', label='Mean')
axes[0].axhline(y=avg_grads.mean().item() + 2 * avg_grads.std().item(), 
               color='g', linestyle='--', label='Threshold (mean + 2σ)')
axes[0].set_xlabel('Gaussian Index')
axes[0].set_ylabel('Average Gradient Norm')
axes[0].set_title('Accumulated Positional Gradients')
axes[0].legend()

# Histogram
axes[1].hist(avg_grads.numpy(), bins=30, edgecolor='black')
axes[1].axvline(x=avg_grads.mean().item() + 2 * avg_grads.std().item(),
               color='g', linestyle='--', linewidth=2, label='Densify threshold')
axes[1].set_xlabel('Average Gradient Norm')
axes[1].set_ylabel('Count')
axes[1].set_title('Gradient Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Gaussians 0-19: avg grad = {avg_grads[:20].mean():.3f} (high - need densification)")
print(f"Gaussians 20-39: avg grad = {avg_grads[20:40].mean():.3f} (low - well-optimized)")
print(f"Gaussians 40-99: avg grad = {avg_grads[40:].mean():.3f} (medium)")

## 3. Splitting Large Gaussians

### When to Split

A Gaussian should be **split** when:
1. It has **high positional gradient** (under-reconstructed)
2. It is **large** (scale > threshold)

### How to Split

Replace one Gaussian with two smaller ones:
- Position: offset along principal axis ± ε
- Scale: reduced by factor (typically 1.6)
- Other properties: copied from parent

In [ ]:
def split_gaussians(
    means: torch.Tensor,      # [N, 3]
    scales: torch.Tensor,     # [N, 3]
    rotations: torch.Tensor,  # [N, 4] quaternions
    opacities: torch.Tensor,  # [N]
    colors: torch.Tensor,     # [N, 3] or [N, K, 3] for SH
    split_mask: torch.Tensor, # [N] boolean
    scale_factor: float = 1.6,
    n_splits: int = 2,
) -> tuple:
    """
    Split large Gaussians with high gradients.
    
    Args:
        means, scales, rotations, opacities, colors: Gaussian parameters
        split_mask: Which Gaussians to split
        scale_factor: How much to reduce scale
        n_splits: Number of children per parent
    
    Returns:
        Tuple of new parameters (means, scales, rotations, opacities, colors)
    """
    N = means.shape[0]
    n_to_split = split_mask.sum().item()
    
    if n_to_split == 0:
        return means, scales, rotations, opacities, colors
    
    # Get indices of Gaussians to split
    split_indices = torch.where(split_mask)[0]
    
    # Sample positions for children (along principal axis)
    # Use scaled covariance to determine offset direction
    parent_scales = scales[split_indices]  # [n_to_split, 3]
    
    # Sample random offsets proportional to scale
    # This places children along the Gaussian's extent
    new_means_list = []
    new_scales_list = []
    new_rotations_list = []
    new_opacities_list = []
    new_colors_list = []
    
    for i in range(n_splits):
        # Random direction based on scale
        samples = torch.randn(n_to_split, 3) * parent_scales
        
        # New positions
        new_means = means[split_indices] + samples
        new_means_list.append(new_means)
        
        # Reduced scales
        new_scales = scales[split_indices] / scale_factor
        new_scales_list.append(new_scales)
        
        # Copy other properties
        new_rotations_list.append(rotations[split_indices].clone())
        new_opacities_list.append(opacities[split_indices].clone())
        new_colors_list.append(colors[split_indices].clone())
    
    # Combine: keep non-split + add new children
    keep_mask = ~split_mask
    
    new_means = torch.cat([means[keep_mask]] + new_means_list, dim=0)
    new_scales = torch.cat([scales[keep_mask]] + new_scales_list, dim=0)
    new_rotations = torch.cat([rotations[keep_mask]] + new_rotations_list, dim=0)
    new_opacities = torch.cat([opacities[keep_mask]] + new_opacities_list, dim=0)
    new_colors = torch.cat([colors[keep_mask]] + new_colors_list, dim=0)
    
    return new_means, new_scales, new_rotations, new_opacities, new_colors


# Demo: split a single Gaussian
means = torch.tensor([[0., 0., 0.]])
scales = torch.tensor([[0.5, 0.3, 0.2]])  # Large in x
rotations = torch.tensor([[1., 0., 0., 0.]])  # Identity
opacities = torch.tensor([0.8])
colors = torch.tensor([[1., 0., 0.]])  # Red

split_mask = torch.tensor([True])

new_means, new_scales, new_rotations, new_opacities, new_colors = split_gaussians(
    means, scales, rotations, opacities, colors, split_mask,
    scale_factor=1.6, n_splits=2
)

print("Before split:")
print(f"  N = {len(means)}")
print(f"  Means: {means}")
print(f"  Scales: {scales}")

print(f"\nAfter split:")
print(f"  N = {len(new_means)}")
print(f"  Means:\n{new_means}")
print(f"  Scales:\n{new_scales}")

In [ ]:
# Visualize splitting in 2D
def visualize_split_2d():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Before: single large Gaussian
    ax = axes[0]
    ax.set_xlim(-2, 2)
    ax.set_ylim(-1.5, 1.5)
    
    # Parent Gaussian (large ellipse)
    parent = Ellipse((0, 0), width=2.0, height=0.8, angle=20,
                    fill=True, alpha=0.5, color='blue', label='Parent')
    ax.add_patch(parent)
    ax.scatter([0], [0], s=100, c='blue', marker='x', zorder=10)
    ax.set_title('Before Split\n(Large Gaussian with high gradient)')
    ax.set_aspect('equal')
    ax.legend()
    
    # After: two smaller Gaussians
    ax = axes[1]
    ax.set_xlim(-2, 2)
    ax.set_ylim(-1.5, 1.5)
    
    # Children (smaller ellipses)
    child1_pos = (-0.5, -0.2)
    child2_pos = (0.5, 0.2)
    scale_reduction = 1.6
    
    child1 = Ellipse(child1_pos, width=2.0/scale_reduction, height=0.8/scale_reduction,
                    angle=20, fill=True, alpha=0.5, color='red', label='Child 1')
    child2 = Ellipse(child2_pos, width=2.0/scale_reduction, height=0.8/scale_reduction,
                    angle=20, fill=True, alpha=0.5, color='green', label='Child 2')
    
    ax.add_patch(child1)
    ax.add_patch(child2)
    ax.scatter([child1_pos[0], child2_pos[0]], [child1_pos[1], child2_pos[1]], 
              s=100, c=['red', 'green'], marker='x', zorder=10)
    
    # Show original outline
    parent_outline = Ellipse((0, 0), width=2.0, height=0.8, angle=20,
                            fill=False, linestyle='--', edgecolor='blue', linewidth=2)
    ax.add_patch(parent_outline)
    
    ax.set_title(f'After Split\n(Scale reduced by {scale_reduction}x, offset along axes)')
    ax.set_aspect('equal')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

visualize_split_2d()

## 4. Cloning Small Gaussians

### When to Clone

A Gaussian should be **cloned** when:
1. It has **high positional gradient** (under-reconstructed)
2. It is **small** (scale < threshold)

### Why Clone Instead of Split?

- Small Gaussians are already at appropriate scale
- We need **more coverage**, not smaller size
- Clone creates duplicate in direction of gradient

In [ ]:
def clone_gaussians(
    means: torch.Tensor,
    scales: torch.Tensor,
    rotations: torch.Tensor,
    opacities: torch.Tensor,
    colors: torch.Tensor,
    clone_mask: torch.Tensor,
    gradient_directions: torch.Tensor = None,
    offset_scale: float = 1.0,
) -> tuple:
    """
    Clone small Gaussians with high gradients.
    
    Args:
        means, scales, rotations, opacities, colors: Gaussian parameters
        clone_mask: Which Gaussians to clone
        gradient_directions: Direction to offset clones (optional)
        offset_scale: How far to offset clones
    
    Returns:
        Tuple of new parameters
    """
    n_to_clone = clone_mask.sum().item()
    
    if n_to_clone == 0:
        return means, scales, rotations, opacities, colors
    
    # Get indices to clone
    clone_indices = torch.where(clone_mask)[0]
    
    # Compute offset for clones
    if gradient_directions is not None:
        # Move in gradient direction
        offsets = gradient_directions[clone_indices] * offset_scale
    else:
        # Random offset proportional to scale
        offsets = torch.randn(n_to_clone, 3) * scales[clone_indices] * offset_scale
    
    # Clone with offset
    clone_means = means[clone_indices] + offsets
    clone_scales = scales[clone_indices].clone()
    clone_rotations = rotations[clone_indices].clone()
    clone_opacities = opacities[clone_indices].clone()
    clone_colors = colors[clone_indices].clone()
    
    # Append clones to original (keep all originals)
    new_means = torch.cat([means, clone_means], dim=0)
    new_scales = torch.cat([scales, clone_scales], dim=0)
    new_rotations = torch.cat([rotations, clone_rotations], dim=0)
    new_opacities = torch.cat([opacities, clone_opacities], dim=0)
    new_colors = torch.cat([colors, clone_colors], dim=0)
    
    return new_means, new_scales, new_rotations, new_opacities, new_colors


# Demo: clone a small Gaussian
means = torch.tensor([[0., 0., 0.]])
scales = torch.tensor([[0.1, 0.08, 0.05]])  # Small
rotations = torch.tensor([[1., 0., 0., 0.]])
opacities = torch.tensor([0.9])
colors = torch.tensor([[0., 0., 1.]])  # Blue

clone_mask = torch.tensor([True])
grad_dirs = torch.tensor([[1., 0.5, 0.]])  # Gradient points to +x, +y

new_means, new_scales, new_rotations, new_opacities, new_colors = clone_gaussians(
    means, scales, rotations, opacities, colors, clone_mask,
    gradient_directions=grad_dirs, offset_scale=0.3
)

print("Before clone:")
print(f"  N = {len(means)}")
print(f"  Means: {means}")

print(f"\nAfter clone:")
print(f"  N = {len(new_means)} (original + clone)")
print(f"  Means:\n{new_means}")
print(f"  Scales (same):\n{new_scales}")

In [ ]:
# Visualize cloning in 2D
def visualize_clone_2d():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Before: single small Gaussian
    ax = axes[0]
    ax.set_xlim(-1, 2)
    ax.set_ylim(-1, 1.5)
    
    original = Ellipse((0, 0), width=0.4, height=0.3, angle=0,
                      fill=True, alpha=0.6, color='blue', label='Original')
    ax.add_patch(original)
    ax.scatter([0], [0], s=100, c='blue', marker='x', zorder=10)
    
    # Show gradient direction
    ax.arrow(0.2, 0.15, 0.5, 0.3, head_width=0.1, head_length=0.05, 
            fc='red', ec='red', linewidth=2)
    ax.text(0.5, 0.55, 'Gradient\ndirection', fontsize=10, color='red')
    
    ax.set_title('Before Clone\n(Small Gaussian with high gradient)')
    ax.set_aspect('equal')
    ax.legend()
    
    # After: original + clone
    ax = axes[1]
    ax.set_xlim(-1, 2)
    ax.set_ylim(-1, 1.5)
    
    original = Ellipse((0, 0), width=0.4, height=0.3, angle=0,
                      fill=True, alpha=0.6, color='blue', label='Original (kept)')
    clone = Ellipse((0.8, 0.5), width=0.4, height=0.3, angle=0,
                   fill=True, alpha=0.6, color='green', label='Clone (new)')
    
    ax.add_patch(original)
    ax.add_patch(clone)
    ax.scatter([0, 0.8], [0, 0.5], s=100, c=['blue', 'green'], marker='x', zorder=10)
    
    # Show movement
    ax.annotate('', xy=(0.8, 0.5), xytext=(0, 0),
               arrowprops=dict(arrowstyle='->', color='gray', linestyle='--'))
    
    ax.set_title('After Clone\n(Original kept, clone moved in gradient direction)')
    ax.set_aspect('equal')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

visualize_clone_2d()

## 5. Pruning Gaussians

### Pruning Criteria

Remove Gaussians that:

1. **Low opacity**: α < ε_α (typically 0.005)
   - Nearly invisible, contribute little to rendering

2. **Too large in world space**: scale > threshold
   - May indicate floaters or outliers

3. **Too large on screen**: 2D extent > image_size / 4
   - Cover too much area, reduce quality

### Opacity Reset

Periodically reset all opacities closer to zero:
- Allows network to "re-evaluate" importance
- Truly unnecessary Gaussians will stay low → pruned

In [ ]:
def prune_gaussians(
    means: torch.Tensor,
    scales: torch.Tensor,
    rotations: torch.Tensor,
    opacities: torch.Tensor,
    colors: torch.Tensor,
    opacity_threshold: float = 0.005,
    scale_threshold: float = None,
    screen_extent: torch.Tensor = None,
    screen_threshold: float = None,
) -> tuple:
    """
    Prune Gaussians based on various criteria.
    
    Args:
        means, scales, rotations, opacities, colors: Gaussian parameters
        opacity_threshold: Remove if opacity < threshold
        scale_threshold: Remove if max(scale) > threshold (world space)
        screen_extent: 2D extent of each Gaussian [N]
        screen_threshold: Remove if screen_extent > threshold
    
    Returns:
        Tuple of pruned parameters and prune statistics
    """
    N = means.shape[0]
    
    # Start with all valid
    keep_mask = torch.ones(N, dtype=torch.bool)
    
    prune_reasons = {
        'low_opacity': 0,
        'large_scale': 0,
        'large_screen': 0,
    }
    
    # Prune low opacity
    low_opacity = opacities < opacity_threshold
    prune_reasons['low_opacity'] = low_opacity.sum().item()
    keep_mask &= ~low_opacity
    
    # Prune large scale
    if scale_threshold is not None:
        max_scale = scales.max(dim=-1).values
        large_scale = max_scale > scale_threshold
        prune_reasons['large_scale'] = large_scale.sum().item()
        keep_mask &= ~large_scale
    
    # Prune large screen extent
    if screen_extent is not None and screen_threshold is not None:
        large_screen = screen_extent > screen_threshold
        prune_reasons['large_screen'] = large_screen.sum().item()
        keep_mask &= ~large_screen
    
    # Apply pruning
    new_means = means[keep_mask]
    new_scales = scales[keep_mask]
    new_rotations = rotations[keep_mask]
    new_opacities = opacities[keep_mask]
    new_colors = colors[keep_mask]
    
    return new_means, new_scales, new_rotations, new_opacities, new_colors, prune_reasons


def reset_opacity(
    opacities: torch.Tensor,
    reset_value: float = 0.01,
) -> torch.Tensor:
    """
    Reset opacities to near-zero value.
    
    This forces the network to re-justify each Gaussian's existence.
    Used periodically during training (e.g., every 3000 iterations).
    """
    # In practice, we work in logit space
    # opacity = sigmoid(opacity_raw)
    # To set opacity ≈ 0.01, we need sigmoid^{-1}(0.01) ≈ -4.6
    reset_logit = torch.log(torch.tensor(reset_value / (1 - reset_value)))
    return torch.full_like(opacities, reset_value)


# Demo: pruning
N = 100
torch.manual_seed(42)

means = torch.randn(N, 3)
scales = torch.rand(N, 3) * 0.5  # Some will be large
scales[:10, 0] = 2.0  # Make first 10 very large
rotations = torch.randn(N, 4)
rotations = rotations / rotations.norm(dim=-1, keepdim=True)

opacities = torch.rand(N)
opacities[10:30] = 0.001  # Make some nearly invisible

colors = torch.rand(N, 3)

# Prune
new_means, new_scales, new_rotations, new_opacities, new_colors, reasons = prune_gaussians(
    means, scales, rotations, opacities, colors,
    opacity_threshold=0.005,
    scale_threshold=1.5,
)

print("Pruning Results:")
print("=" * 50)
print(f"Original: {N} Gaussians")
print(f"After pruning: {len(new_means)} Gaussians")
print(f"\nPruned by reason:")
for reason, count in reasons.items():
    print(f"  {reason}: {count}")
print(f"\nTotal removed: {N - len(new_means)}")

In [ ]:
# Visualize pruning criteria
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Opacity distribution
ax = axes[0]
ax.hist(opacities.numpy(), bins=30, edgecolor='black', alpha=0.7)
ax.axvline(x=0.005, color='red', linestyle='--', linewidth=2, label='Threshold')
ax.fill_betweenx([0, ax.get_ylim()[1]], 0, 0.005, alpha=0.3, color='red', label='Pruned region')
ax.set_xlabel('Opacity')
ax.set_ylabel('Count')
ax.set_title('Opacity Distribution\n(Prune if < 0.005)')
ax.legend()

# Scale distribution
ax = axes[1]
max_scales = scales.max(dim=-1).values
ax.hist(max_scales.numpy(), bins=30, edgecolor='black', alpha=0.7)
ax.axvline(x=1.5, color='red', linestyle='--', linewidth=2, label='Threshold')
ax.fill_betweenx([0, ax.get_ylim()[1]], 1.5, ax.get_xlim()[1], alpha=0.3, color='red', label='Pruned region')
ax.set_xlabel('Max Scale')
ax.set_ylabel('Count')
ax.set_title('Scale Distribution\n(Prune if > 1.5)')
ax.legend()

# Combined visualization
ax = axes[2]
# Color by kept/pruned
keep_mask = (opacities >= 0.005) & (max_scales <= 1.5)
colors_plot = ['green' if k else 'red' for k in keep_mask.numpy()]

ax.scatter(opacities.numpy(), max_scales.numpy(), c=colors_plot, alpha=0.6, s=50)
ax.axhline(y=1.5, color='red', linestyle='--', linewidth=2)
ax.axvline(x=0.005, color='red', linestyle='--', linewidth=2)

# Legend
kept_patch = mpatches.Patch(color='green', label=f'Kept ({keep_mask.sum().item()})')
pruned_patch = mpatches.Patch(color='red', label=f'Pruned ({(~keep_mask).sum().item()})')
ax.legend(handles=[kept_patch, pruned_patch])

ax.set_xlabel('Opacity')
ax.set_ylabel('Max Scale')
ax.set_title('Pruning Decision Space')

plt.tight_layout()
plt.show()

## 6. Complete Densification Controller

Let's combine everything into a complete densification controller.

In [ ]:
class DensificationController:
    """
    Controls adaptive density for 3DGS training.
    
    Handles:
    - Gradient accumulation
    - Splitting large Gaussians
    - Cloning small Gaussians
    - Pruning unnecessary Gaussians
    - Opacity reset
    """
    
    def __init__(
        self,
        grad_threshold: float = 0.0002,
        split_scale_threshold: float = 0.01,  # % of scene extent
        opacity_threshold: float = 0.005,
        max_screen_size: float = 0.1,  # % of image
        densify_interval: int = 100,
        opacity_reset_interval: int = 3000,
        densify_start: int = 500,
        densify_end: int = 15000,
    ):
        self.grad_threshold = grad_threshold
        self.split_scale_threshold = split_scale_threshold
        self.opacity_threshold = opacity_threshold
        self.max_screen_size = max_screen_size
        self.densify_interval = densify_interval
        self.opacity_reset_interval = opacity_reset_interval
        self.densify_start = densify_start
        self.densify_end = densify_end
        
        # Statistics
        self.stats = {
            'n_split': 0,
            'n_cloned': 0,
            'n_pruned': 0,
        }
    
    def should_densify(self, iteration: int) -> bool:
        """Check if should perform densification at this iteration."""
        if iteration < self.densify_start or iteration > self.densify_end:
            return False
        return iteration % self.densify_interval == 0
    
    def should_reset_opacity(self, iteration: int) -> bool:
        """Check if should reset opacity at this iteration."""
        return iteration > 0 and iteration % self.opacity_reset_interval == 0
    
    def densify_and_prune(
        self,
        means: torch.Tensor,
        scales: torch.Tensor,
        rotations: torch.Tensor,
        opacities: torch.Tensor,
        colors: torch.Tensor,
        avg_gradients: torch.Tensor,
        scene_extent: float = 1.0,
    ) -> tuple:
        """
        Perform full densification and pruning.
        
        Args:
            means, scales, rotations, opacities, colors: Gaussian parameters
            avg_gradients: Average positional gradients [N]
            scene_extent: Spatial extent of scene (for scale threshold)
        
        Returns:
            Updated parameters
        """
        N_before = means.shape[0]
        
        # Identify Gaussians to densify
        high_grad = avg_gradients > self.grad_threshold
        
        # Large Gaussians: need splitting
        max_scale = scales.max(dim=-1).values
        scale_thresh = self.split_scale_threshold * scene_extent
        is_large = max_scale > scale_thresh
        
        split_mask = high_grad & is_large
        clone_mask = high_grad & ~is_large
        
        # Step 1: Split large Gaussians
        means, scales, rotations, opacities, colors = split_gaussians(
            means, scales, rotations, opacities, colors, split_mask
        )
        self.stats['n_split'] += split_mask.sum().item()
        
        # Update clone mask for new size
        # (After split, some indices changed)
        n_kept = (~split_mask).sum().item()
        clone_mask_new = torch.zeros(len(means), dtype=torch.bool)
        clone_mask_new[:n_kept] = clone_mask[~split_mask]
        
        # Step 2: Clone small Gaussians
        means, scales, rotations, opacities, colors = clone_gaussians(
            means, scales, rotations, opacities, colors, clone_mask_new
        )
        self.stats['n_cloned'] += clone_mask.sum().item()
        
        # Step 3: Prune
        means, scales, rotations, opacities, colors, reasons = prune_gaussians(
            means, scales, rotations, opacities, colors,
            opacity_threshold=self.opacity_threshold,
            scale_threshold=scale_thresh * 10,  # Very large
        )
        self.stats['n_pruned'] += sum(reasons.values())
        
        N_after = means.shape[0]
        
        return means, scales, rotations, opacities, colors
    
    def print_stats(self):
        """Print densification statistics."""
        print(f"Densification Statistics:")
        print(f"  Split: {self.stats['n_split']}")
        print(f"  Cloned: {self.stats['n_cloned']}")
        print(f"  Pruned: {self.stats['n_pruned']}")


# Test the controller
controller = DensificationController(
    grad_threshold=0.5,
    split_scale_threshold=0.3,
    opacity_threshold=0.005,
)

# Create test data
N = 50
torch.manual_seed(42)

means = torch.randn(N, 3)
scales = torch.rand(N, 3) * 0.5
scales[:5] = 0.6  # Large ones
rotations = torch.randn(N, 4)
rotations = rotations / rotations.norm(dim=-1, keepdim=True)
opacities = torch.rand(N)
opacities[40:] = 0.001  # Low opacity
colors = torch.rand(N, 3)

# Simulate high gradients for some
avg_gradients = torch.rand(N) * 0.3
avg_gradients[:10] = 1.0  # High gradient

print(f"Before densification: {N} Gaussians")

# Densify
new_means, new_scales, new_rotations, new_opacities, new_colors = controller.densify_and_prune(
    means, scales, rotations, opacities, colors, avg_gradients, scene_extent=1.0
)

print(f"After densification: {len(new_means)} Gaussians")
print()
controller.print_stats()

## 7. Visualizing Densification Evolution

Let's simulate how Gaussians evolve during training with adaptive control.

In [ ]:
def simulate_training_with_densification(n_iterations=100):
    """
    Simulate simplified 3DGS training with densification.
    """
    # Initial Gaussians (sparse)
    torch.manual_seed(42)
    N = 20
    
    means = torch.randn(N, 2) * 0.5  # 2D for visualization
    scales = torch.rand(N, 2) * 0.15 + 0.05
    opacities = torch.rand(N) * 0.5 + 0.4
    
    history = [{
        'iter': 0,
        'N': N,
        'means': means.clone(),
        'scales': scales.clone(),
        'opacities': opacities.clone(),
    }]
    
    controller = DensificationController(
        grad_threshold=0.3,
        split_scale_threshold=0.1,
        opacity_threshold=0.1,
        densify_interval=20,
        densify_start=10,
        densify_end=80,
    )
    
    for i in range(1, n_iterations + 1):
        N = means.shape[0]
        
        # Simulate gradients (high for some random Gaussians)
        avg_gradients = torch.rand(N) * 0.2
        # Some random ones have high gradient
        high_grad_idx = torch.randperm(N)[:max(1, N // 5)]
        avg_gradients[high_grad_idx] = torch.rand(len(high_grad_idx)) * 0.5 + 0.5
        
        # Simulate optimization step (slight movement)
        means = means + torch.randn_like(means) * 0.01
        scales = scales * (1 + torch.randn_like(scales) * 0.05)
        opacities = torch.clamp(opacities + torch.randn(N) * 0.02, 0.01, 0.99)
        
        # Densification
        if controller.should_densify(i):
            # Add 3D dummy dimensions for API compatibility
            means_3d = torch.cat([means, torch.zeros(N, 1)], dim=-1)
            scales_3d = torch.cat([scales, scales.mean(dim=-1, keepdim=True)], dim=-1)
            rotations = torch.zeros(N, 4)
            rotations[:, 0] = 1.0
            colors = torch.rand(N, 3)
            
            means_3d, scales_3d, rotations, opacities, colors = controller.densify_and_prune(
                means_3d, scales_3d, rotations, opacities, colors,
                avg_gradients, scene_extent=1.0
            )
            
            # Extract 2D
            means = means_3d[:, :2]
            scales = scales_3d[:, :2]
            
            history.append({
                'iter': i,
                'N': len(means),
                'means': means.clone(),
                'scales': scales.clone(),
                'opacities': opacities.clone(),
            })
    
    return history, controller.stats


history, final_stats = simulate_training_with_densification(100)

print(f"Training simulation complete!")
print(f"Snapshots: {len(history)}")
print(f"Final Gaussian count: {history[-1]['N']}")
print(f"\nFinal stats:")
for k, v in final_stats.items():
    print(f"  {k}: {v}")

In [ ]:
# Visualize evolution
n_snapshots = min(6, len(history))
indices = [int(i * (len(history) - 1) / (n_snapshots - 1)) for i in range(n_snapshots)]

fig, axes = plt.subplots(2, n_snapshots, figsize=(4 * n_snapshots, 7))

for col, idx in enumerate(indices):
    h = history[idx]
    
    # Top: Gaussian visualization
    ax = axes[0, col]
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    
    for i in range(h['N']):
        ellipse = Ellipse(
            (h['means'][i, 0].item(), h['means'][i, 1].item()),
            width=h['scales'][i, 0].item() * 4,
            height=h['scales'][i, 1].item() * 4,
            angle=0,
            fill=True,
            alpha=h['opacities'][i].item() * 0.5,
            color=plt.cm.viridis(h['opacities'][i].item()),
        )
        ax.add_patch(ellipse)
    
    ax.set_title(f"Iter {h['iter']}\nN = {h['N']}")
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    # Bottom: Opacity histogram
    ax = axes[1, col]
    ax.hist(h['opacities'].numpy(), bins=15, range=(0, 1), edgecolor='black', alpha=0.7)
    ax.set_xlabel('Opacity')
    ax.set_ylabel('Count')
    ax.set_title(f'Opacity Distribution')

plt.suptitle('Gaussian Evolution During Training with Adaptive Control', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Plot N over time
fig, ax = plt.subplots(figsize=(10, 4))
iters = [h['iter'] for h in history]
n_gaussians = [h['N'] for h in history]

ax.plot(iters, n_gaussians, 'b-o', markersize=5)
ax.set_xlabel('Iteration')
ax.set_ylabel('Number of Gaussians')
ax.set_title('Gaussian Population Over Training')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Official 3DGS Schedule

Here's the densification schedule used in the official 3DGS implementation.

In [ ]:
# Official 3DGS densification schedule
print("Official 3DGS Densification Schedule:")
print("=" * 60)
print("""
Training iterations: 30,000

Densification:
  - Start: iteration 500
  - End: iteration 15,000
  - Interval: every 100 iterations
  - Gradient threshold: 0.0002
  
Opacity reset:
  - Every 3,000 iterations (during densification period)
  - Reset opacity to ~0.01

Pruning:
  - After each densification step
  - Remove if: opacity < 0.005
  - Remove if: too large on screen (> 0.1 of image)
  - Remove if: scale > world threshold

Split vs Clone decision:
  - If scale > scene_extent * 0.01: SPLIT
  - If scale <= scene_extent * 0.01: CLONE
""")

# Visualize schedule
fig, ax = plt.subplots(figsize=(14, 4))

iterations = np.arange(0, 30001, 100)

# Mark different phases
ax.axvspan(0, 500, alpha=0.3, color='gray', label='Warmup (no densification)')
ax.axvspan(500, 15000, alpha=0.3, color='green', label='Densification active')
ax.axvspan(15000, 30000, alpha=0.3, color='blue', label='Refinement only')

# Mark opacity resets
for reset_iter in [3000, 6000, 9000, 12000, 15000]:
    ax.axvline(x=reset_iter, color='red', linestyle='--', alpha=0.7)
ax.axvline(x=3000, color='red', linestyle='--', alpha=0.7, label='Opacity reset')

# Mark densification intervals
for i in range(500, 15001, 100):
    ax.axvline(x=i, color='green', alpha=0.1, linewidth=0.5)

ax.set_xlim(0, 30000)
ax.set_xlabel('Iteration')
ax.set_title('3DGS Training Schedule')
ax.legend(loc='upper right')
ax.set_yticks([])

plt.tight_layout()
plt.show()

## 9. Summary: Adaptive Density Control

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Gradient criterion** | High positional gradient = needs densification |
| **Splitting** | Large Gaussians → 2 smaller children |
| **Cloning** | Small Gaussians → duplicate in gradient direction |
| **Pruning** | Remove low-opacity, oversized Gaussians |
| **Opacity reset** | Periodically reset to re-evaluate importance |

### The Algorithm

```python
for iteration in range(max_iterations):
    # Forward & backward pass
    loss = render_and_compare()
    loss.backward()
    
    # Accumulate gradients
    accumulator.add(means_2d.grad)
    
    # Densification (every N iterations)
    if should_densify(iteration):
        avg_grad = accumulator.get_average()
        
        # Split large Gaussians with high gradient
        split_mask = (avg_grad > threshold) & (scale > scale_threshold)
        split(gaussians[split_mask])
        
        # Clone small Gaussians with high gradient
        clone_mask = (avg_grad > threshold) & (scale <= scale_threshold)
        clone(gaussians[clone_mask])
        
        # Prune
        prune(opacity < opacity_threshold)
        
        accumulator.reset()
    
    # Opacity reset (every M iterations)
    if should_reset_opacity(iteration):
        reset_all_opacities()
```

---

## Key Takeaways

1. Adaptive control is crucial for quality reconstruction
2. Gradients indicate where more detail is needed
3. Split vs Clone depends on Gaussian size
4. Pruning removes waste and improves efficiency
5. Opacity reset helps identify truly important Gaussians

---

## Next Steps

In the next notebook, we'll put everything together into a complete **Training Pipeline**:

**[08_training_pipeline.ipynb](./08_training_pipeline.ipynb)** - Complete 3DGS Training Loop